## Task 4: General Health Query Chatbot (Prompt Engineering Based)



### 1. Install Necessary Libraries



In [1]:
pip install --upgrade huggingface_hub transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


### 2. Set Up Hugging Face API Token





In [2]:
# Used to securely store your API key
from google.colab import userdata
import os

# Retrieve the Hugging Face token from Colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

print("Hugging Face token loaded.")

Hugging Face token loaded.


### 3. Load the LLM

We will use `Mistral-7B-Instruct-v0.2`

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Ensure GPU is available
if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required to run this model efficiently. Please enable a GPU runtime in Colab (Runtime -> Change runtime type -> T4 GPU).")

# Configure quantization for efficient loading
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, # Changed to 4-bit quantization for better memory efficiency
    bnb_4bit_quant_type="nf4", # Recommended quantization type for 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True, # Further optimize quantization
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto" # Keep auto; 4-bit should help it fit
)

print(f"Model '{model_name}' loaded successfully using 4-bit quantization.")

# Optionally print GPU memory usage
if torch.cuda.is_available():
    print(f"Current GPU memory usage: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model 'mistralai/Mistral-7B-Instruct-v0.2' loaded successfully using 4-bit quantization.
Current GPU memory usage: 3.85 GB
Total GPU memory: 14.56 GB


### 4. Create the Chatbot Function with Prompt Engineering and Safety Filters



In [4]:
from transformers import pipeline

# Create a text generation pipeline
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

def get_health_response(user_query):
    # System prompt for prompt engineering
    system_prompt = "Act like a helpful medical assistant. Provide clear, concise, and general information. DO NOT give specific medical advice or diagnoses. Always recommend consulting a healthcare professional for personalized guidance."

    # Combine system prompt and user query
    messages = [
        {"role": "user", "content": system_prompt + "\n\n" + user_query}
    ]

    # Apply the tokenizer's chat template to format the prompt
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Generate response
    sequences = pipeline(
        prompt,
        max_new_tokens=200, # Limit response length
        do_sample=True, # Enable sampling for more varied responses
        temperature=0.7, # Adjust creativity
        top_k=50, # Limit the number of highest probability vocabulary tokens to consider
        top_p=0.95, # Nucleus sampling: only consider tokens from the smallest set of most probable tokens whose cumulative probability exceeds p
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_text = sequences[0]['generated_text']

    # Extract only the assistant's response part (after the last [/INST])
    response_start_index = generated_text.rfind('[/INST]')
    if response_start_index != -1:
        assistant_response = generated_text[response_start_index + len('[/INST]'):].strip()
    else:
        assistant_response = generated_text # Fallback if tag is not found

    # Add a final safety disclaimer (even if the model might include it)
    safety_disclaimer = "\n\n**Disclaimer:** This information is for general knowledge and informational purposes only, and does not constitute medical advice. Always consult a qualified healthcare professional for any health concerns or before making any decisions related to your health or treatment."

    return assistant_response + safety_disclaimer

print("Chatbot function 'get_health_response' defined.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Chatbot function 'get_health_response' defined.


### 5. Test the Chatbot

Let's try some example queries.

In [5]:
# Example query 1
query1 = "What causes a sore throat?"
print(f"User: {query1}")
response1 = get_health_response(query1)
print(f"Bot: {response1}\n")

# Example query 2
query2 = "Is paracetamol safe for children?"
print(f"User: {query2}")
response2 = get_health_response(query2)
print(f"Bot: {response2}\n")

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'eos_token_id', 'do_sample', 'max_new_tokens', 'temperature', 'top_p', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


User: What causes a sore throat?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for ope

Bot: A sore throat is a common condition characterized by pain, inflammation, and sometimes swelling in the throat. It can be caused by various factors, including:

1. Viral infections: The most common cause of a sore throat is a viral infection, such as the common cold or flu.
2. Bacterial infections: Bacterial infections, such as strep throat (caused by Streptococcus bacteria), can also cause a sore throat.
3. Allergies: Allergies to pollen, dust, or other environmental factors can cause a sore throat.
4. Irritants: Irritants such as smoking, acid reflux, and dry air can cause a sore throat.
5. Injury: Injury to the throat, such as from singing, shouting, or forceful vomiting, can cause a sore throat.

Symptoms of a sore throat may

**Disclaimer:** This information is for general knowledge and informational purposes only, and does not constitute medical advice. Always consult a qualified healthcare professional for any health concerns or before making any decisions related to your he